Celda 1: Instalación de Dependencias

Instalamos las librerías necesarias de LangChain, LangGraph, el conector de Mistral, la base de datos vectorial en memoria (FAISS) y herramientas de procesamiento de texto.

**Nota importante: Después de ejecutar la celda de instalación por primera vez, por favor reinicie la sesión manualmente desde el menú superior para aplicar los cambios de versiones y vuelva a ejecutar la celda** (Entorno de ejecución > Reiniciar sesión)

In [ ]:
# 1. Instalamos las librerías principales del proyecto (añadimos langgraph y tabulate)
!pip install --quiet langchain-mistralai langchain-core langchain-community faiss-cpu pandas unicodedata2 langgraph tabulate

# 2. Forzamos la versión 2.32.5 que satisface a LangChain y es perfectamente compatible con Colab
!pip install --quiet requests==2.32.5

Celda 2: Configuración de la API Key y Variables de Entorno

Configuramos tu clave de Mistral para que los modelos de lenguaje y de embeddings puedan autenticarse.

In [ ]:
import os
from google.colab import userdata

# Configuración de la API Key provista
os.environ["MISTRAL_API_KEY"] = userdata.get('MISTRAL_API_KEY')
print("Clave de API de Mistral configurada correctamente.")

Clave de API de Mistral configurada correctamente.


Celda 3: Carga, Normalización y Copia del Dataset

Cargamos el archivo, creamos una copia explícita para conservar el original intacto, y normalizamos el texto del DataFrame de trabajo (pasando a minúsculas, eliminando espacios extras y removiendo tildes/acentos).

In [ ]:
import pandas as pd
import unicodedata

# 1. Cargar el dataset original
df_all = pd.read_csv("healthcare_dataset.csv")

# 2. Tomar las primeras 1000 filas para que la API de Mistral responda en segundos
df_original = df_all.head(1000).copy()

# 3. Crear la copia explícita para trabajar la normalización
df_normalizado = df_original.copy()

# Función auxiliar para limpiar y normalizar texto
def normalizar_texto(texto):
    if pd.isna(texto):
        return ""
    texto_str = str(texto).lower().strip()
    texto_sin_tildes = ''.join(
        c for c in unicodedata.normalize('NFD', texto_str)
        if unicodedata.category(c) != 'Mn'
    )
    return texto_sin_tildes

# 4. Aplicar la normalización
for columna in df_normalizado.columns:
    df_normalizado[columna] = df_normalizado[columna].apply(normalizar_texto)

df_normalizado.fillna("no disponible", inplace=True)

print("¡Dataset optimizado y normalizado con éxito!")
print(f"Filas reducidas para la práctica: {len(df_normalizado)} (¡Ahora correrá superrápido!)")
df_normalizado.head(3)

¡Dataset optimizado y normalizado con éxito!
Filas reducidas para la práctica: 1000 (¡Ahora correrá superrápido!)


,Name,Age,Gender,Blood Type,Medical Condition,Date of Admission,Doctor,Hospital,Insurance Provider,Billing Amount,Room Number,Admission Type,Discharge Date,Medication,Test Results
0,bobby jackson,30,male,b-,cancer,2024-01-31,matthew smith,sons and miller,blue cross,18856.281305978155,328,urgent,2024-02-02,paracetamol,normal
1,leslie terry,62,male,a+,obesity,2019-08-20,samantha davies,kim inc,medicare,33643.327286577885,265,emergency,2019-08-26,ibuprofen,inconclusive
2,danny smith,76,female,a-,obesity,2022-09-22,tiffany mitchell,cook plc,aetna,27955.096078842456,205,emergency,2022-10-07,aspirin,normal


Celda 3.1: Informe del dataset normalizado

In [ ]:
print("="*60)
print("   INFORME TÉCNICO DEL DATASET NORMALIZADO (PRIMERAS 1000 FILAS)")
print("="*60)

# 1. Estructura general y Tipos de Datos por columna
print("\n[1] ESTRUCTURA Y TIPOS DE DATOS:")
df_normalizado.info()

print("\n" + "="*60)

# 2. Conteo de valores nulos para asegurar la limpieza
print("\n[2] CONTROL DE VALORES NULOS POR COLUMNA:")
print(df_normalizado.isnull().sum())

print("\n" + "="*60)

# 3. Muestra visual de las primeras 3 filas
print("\n[3] VISTA PREVIA DE LOS DATOS NORMALIZADOS:")
df_normalizado.head(3)

   INFORME TÉCNICO DEL DATASET NORMALIZADO (PRIMERAS 1000 FILAS)

[1] ESTRUCTURA Y TIPOS DE DATOS:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1000 entries, 0 to 999
Data columns (total 15 columns):
 #   Column              Non-Null Count  Dtype 
---  ------              --------------  ----- 
 0   Name                1000 non-null   object
 1   Age                 1000 non-null   object
 2   Gender              1000 non-null   object
 3   Blood Type          1000 non-null   object
 4   Medical Condition   1000 non-null   object
 5   Date of Admission   1000 non-null   object
 6   Doctor              1000 non-null   object
 7   Hospital            1000 non-null   object
 8   Insurance Provider  1000 non-null   object
 9   Billing Amount      1000 non-null   object
 10  Room Number         1000 non-null   object
 11  Admission Type      1000 non-null   object
 12  Discharge Date      1000 non-null   object
 13  Medication          1000 non-null   object
 14  Test Results        10

,Name,Age,Gender,Blood Type,Medical Condition,Date of Admission,Doctor,Hospital,Insurance Provider,Billing Amount,Room Number,Admission Type,Discharge Date,Medication,Test Results
0,bobby jackson,30,male,b-,cancer,2024-01-31,matthew smith,sons and miller,blue cross,18856.281305978155,328,urgent,2024-02-02,paracetamol,normal
1,leslie terry,62,male,a+,obesity,2019-08-20,samantha davies,kim inc,medicare,33643.327286577885,265,emergency,2019-08-26,ibuprofen,inconclusive
2,danny smith,76,female,a-,obesity,2022-09-22,tiffany mitchell,cook plc,aetna,27955.096078842456,205,emergency,2022-10-07,aspirin,normal


Celda Extra(Opcional): Descargar el dataset normalizado

In [ ]:
from google.colab import files

# 1. Guardar el DataFrame normalizado como un archivo CSV en el entorno de Colab
nombre_archivo = "healthcare_dataset_normalizado.csv"
df_normalizado.to_csv(nombre_archivo, index=False, encoding='utf-8')

print(f"Archivo '{nombre_archivo}' generado en el entorno.")
print("Iniciando la descarga en tu navegador...")

# 2. Forzar la descarga automática al almacenamiento local de tu computadora
files.download(nombre_archivo)

Archivo 'healthcare_dataset_normalizado.csv' generado en el entorno.
Iniciando la descarga en tu navegador...


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

Celda 4: Transformación de Datos a Documentos y Creación del Índice RAG (Vectorstore)

Convertimos las filas del DataFrame en objetos Document y los indexamos en una base de datos vectorial en memoria (FAISS) utilizando Mistral Embeddings.

In [ ]:
from langchain_core.documents import Document
from langchain_community.vectorstores import FAISS
from langchain_mistralai import MistralAIEmbeddings

documentos = []

# Usamos los nombres de columnas originales con mayúscula inicial
for index, row in df_normalizado.iterrows():
    contenido_fila = (
        f"paciente: {row['Name']}. edad: {row['Age']}. genero: {row['Gender']}. "
        f"tipo de sangre: {row['Blood Type']}. condicion medica: {row['Medical Condition']}. "
        f"fecha de admision: {row['Date of Admission']}. doctor: {row['Doctor']}. "
        f"hospital: {row['Hospital']}. aseguradora: {row['Insurance Provider']}. "
        f"monto de facturacion: {row['Billing Amount']}. numero de habitacion: {row['Room Number']}. "
        f"tipo de admision: {row['Admission Type']}. fecha de alta: {row['Discharge Date']}. "
        f"medicacion: {row['Medication']}. resultados de la prueba: {row['Test Results']}."
    )
    doc = Document(page_content=contenido_fila, metadata={"fila": index, "paciente": row['Name']})
    documentos.append(doc)

# Inicializar embeddings de Mistral e índice FAISS
embeddings = MistralAIEmbeddings(model="mistral-embed")
vectorstore = FAISS.from_documents(documentos, embeddings)
retriever = vectorstore.as_retriever(search_kwargs={"k": 5})

print(f"¡Índice RAG creado exitosamente con {len(documentos)} registros!")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


¡Índice RAG creado exitosamente con 1000 registros!


Celda 5: Definición de la Herramienta de Búsqueda (Tool) para el Agente

Encapsulamos el recuperador RAG en una herramienta estructurada, usando un docstring descriptivo para que el agente entienda cuándo invocar la búsqueda.

In [ ]:
from langchain_core.tools import tool

@tool
def buscar_informacion_salud(query: str) -> str:
    """Busca y recupera expedientes, registros médicos, pacientes, doctores e historial clínico dentro del dataset de salud."""
    docs = retriever.invoke(query)
    # Unimos los resultados encontrados separados por saltos de línea
    resultados = "\n\n".join([d.page_content for d in docs])
    return resultados

# Lista de herramientas que tendrá disponibles el Agente
tools = [buscar_informacion_salud]
print("Herramienta de búsqueda RAG registrada.")

Herramienta de búsqueda RAG registrada.


Celda 6: Creación del Agente con LangGraph

Inicializamos el modelo Mistral Large y compilamos el agente dinámico bajo la arquitectura de LangGraph, definiendo sus reglas con un SystemMessage.

In [ ]:
from langchain_mistralai import ChatMistralAI
from langgraph.prebuilt import create_react_agent
from langchain_core.messages import SystemMessage

# 2. Inicializar el modelo Mistral Large
llm = ChatMistralAI(model="mistral-large-latest", temperature=0)

# 3. Definimos las instrucciones usando la estructura estricta de SystemMessage
instrucciones = SystemMessage(
    content=(
        "Eres un asistente médico experto. Tu único objetivo es responder la pregunta del usuario. "
        "Para hacerlo, debes usar OBLIGATORIAMENTE la herramienta 'buscar_informacion_salud' pasando el nombre del paciente o término clave. "
        "Una vez que la herramienta te devuelva la información, léela y redacta tu respuesta final en español de forma breve."
    )
)

# 4. Creamos el agente pasando el SystemMessage correctamente
agent_executor = create_react_agent(llm, tools=tools, prompt=instrucciones)

print("¡Agente de LangGraph reconfigurado correctamente con SystemMessage!")

¡Agente de LangGraph reconfigurado correctamente con SystemMessage!


/usr/local/lib/python3.12/dist-packages/langgraph/checkpoint/serde/encrypted.py:5: LangChainPendingDeprecationWarning: The default value of `allowed_objects` will change in a future version. Pass an explicit value (e.g., allowed_objects='messages' or allowed_objects='core') to suppress this warning.
  from langgraph.checkpoint.serde.jsonplus import JsonPlusSerializer
/tmp/ipykernel_17340/2100502899.py:21: LangGraphDeprecatedSinceV10: create_react_agent has been moved to `langchain.agents`. Please update your import to `from langchain.agents import create_agent`. Deprecated in LangGraph V1.0 to be removed in V2.0.
  agent_executor = create_react_agent(llm, tools=tools, prompt=instrucciones)


Celda 7: Ejecución y Pruebas del Sistema

In [ ]:
# 1. Definir la consulta basándonos en los datos normalizados (recuerda que pasamos todo a minúsculas)
consulta = "¿que resultados de prueba tiene el paciente bobby jackson y que doctor lo atendio?"

print(f"Pregunta enviada al Agente: {consulta}\n")

# 2. Invocamos al agente enviando la estructura de mensajes requerida por LangGraph
resultado = agent_executor.invoke({"messages": [("user", consulta)]})

# 3. Extraemos e imprimimos el contenido del último mensaje del grafo (la respuesta final)
print("--- PROCESO DEL AGENTE ---")
print("\n--- RESPUESTA FINAL ---")
print(resultado["messages"][-1].content)

Pregunta enviada al Agente: ¿que resultados de prueba tiene el paciente bobby jackson y que doctor lo atendio?

--- PROCESO DEL AGENTE ---

--- RESPUESTA FINAL ---
**Resultados de prueba del paciente Bobby Jackson:**
- **Resultados:** Normales.
- **Doctor que lo atendió:** Matthew Smith.


Nota: Ejecutar las celdas de Preguntas Complejas en un intervalo de tiempo (10seg o más de espera entre celdas aprox) para evitar el rate limit de la API_Key de Mistral, caso contrario elegir y ejecutar una única celda

Celda 7.1 (Pregunta Compleja 1)

In [ ]:
import time

consulta_1 = "¿Qué paciente de género masculino tiene asignada la medicación 'ibuprofen' por una condición de 'obesity' y cuál fue su doctor?"
print(f"Ejecutando Pregunta 1: {consulta_1}\n...")

resultado_1 = agent_executor.invoke({"messages": [("user", consulta_1.lower())]})
print("\n--- RESPUESTA FINAL ---")
print(resultado_1["messages"][-1].content)

Ejecutando Pregunta 1: ¿Qué paciente de género masculino tiene asignada la medicación 'ibuprofen' por una condición de 'obesity' y cuál fue su doctor?
...

--- RESPUESTA FINAL ---
El paciente masculino con la condición de **obesity** y medicación **ibuprofen** es:

- **Anthony Hall**
  - **Doctor asignado:** Matthew Williams.


Celda 7.2 (Pregunta Compleja 2)

In [ ]:
consulta_2 = "De los pacientes con diagnóstico de 'cancer', identifica uno que tenga un monto de facturacion alto (mayor a 15000) y dime en qué hospital se encuentra."
print(f"Ejecutando Pregunta 2: {consulta_2}\n...")

resultado_2 = agent_executor.invoke({"messages": [("user", consulta_2.lower())]})
print("\n--- RESPUESTA FINAL ---")
print(resultado_2["messages"][-1].content)

Ejecutando Pregunta 2: De los pacientes con diagnóstico de 'cancer', identifica uno que tenga un monto de facturacion alto (mayor a 15000) y dime en qué hospital se encuentra.
...

--- RESPUESTA FINAL ---
El paciente **David Montoya**, con un monto de facturación de **$37,194.26**, se encuentra en el hospital **Rivera-Peck**.


Celda 7.3 (Pregunta Compleja 3)

In [ ]:
consulta_3 = "Busca un paciente cuya condición médica sea 'diabetes' y su tipo de admision haya sido 'urgent'. ¿Cuáles fueron sus fechas exactas de admision y de alta?"
print(f"Ejecutando Pregunta 3: {consulta_3}\n...")

resultado_3 = agent_executor.invoke({"messages": [("user", consulta_3.lower())]})
print("\n--- RESPUESTA FINAL ---")
print(resultado_3["messages"][-1].content)

Ejecutando Pregunta 3: Busca un paciente cuya condición médica sea 'diabetes' y su tipo de admision haya sido 'urgent'. ¿Cuáles fueron sus fechas exactas de admision y de alta?
...

--- RESPUESTA FINAL ---
Los pacientes con **diabetes** y **tipo de admisión urgente** son los siguientes, junto con sus fechas de admisión y alta:

1. **Jamie Salazar**
   - **Fecha de admisión:** 2020-12-29
   - **Fecha de alta:** 2021-01-07

2. **Anne Anthony**
   - **Fecha de admisión:** 2023-01-20
   - **Fecha de alta:** 2023-01-29
